In [1]:
import os
import sys
import io
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
if os.name == 'nt':
  print("This message is printed with utf-8 encoding. Special characters should display correctly: äöüß")

This message is printed with utf-8 encoding. Special characters should display correctly: äöüß


In [3]:
def load_data(data_dir, img_size=(64,64)):
  images=[]
  labels=[]
  class_names = os.listdir(data_dir)

  for class_idx, class_name in enumerate(class_names):
    class_folder = os.path.join(data_dir, class_name)
    for img_name in os.listdir(class_folder):
      img_path = os.path.join(class_folder, img_name)
      img = cv2.imread(img_path)
      img = cv2.resize(img, img_size)
      images.append(img)
      labels.append(class_idx)
  
  images = np.array(images) / 255.0
  labels = to_categorical(np.array(labels), num_classes=len(class_names))
  return images, labels, class_names

In [4]:
data_dir = r"E:\Jupyter projects\Datasets\ISL\Indian"
images, labels, class_names =load_data(data_dir)

MemoryError: Unable to allocate 3.91 GiB for an array with shape (42745, 64, 64, 3) and data type float64

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(images, labels, test_size=0.2, random_state=42)

In [ ]:
def create_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(64, (3, 3), activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(256, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model
model = create_model(input_shape=(64, 64, 3), num_classes=len(class_names))

In [ ]:
# Step 4: Set up image data generator with adjusted augmentation parameters
datagen = ImageDataGenerator(rotation_range=10,
                             width_shift_range=0.1,
                             height_shift_range=0.1,
                             horizontal_flip=True)

# Step 5: Callbacks for saving best model and early stopping
callbacks = [
    EarlyStopping(patience=5, monitor='val_loss', mode='min', verbose=1),
    ModelCheckpoint('best_model.keras', monitor='val_loss', mode='min', save_best_only=True, verbose=1)
]

# Step 6: Train the model with more epochs
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    epochs=5,  # Increased epochs
                    validation_data=(X_test, y_test),
                    callbacks=callbacks)

# Step 7: Initialize MediaPipe Hands for real-time hand detection
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)

In [ ]:
# Step 8:
def predict_sign(frame, model, class_names):
    img = cv2.resize(frame, (64, 64))  # Resize to the input size of the model
    img = img / 255.0  # Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    prediction = model.predict(img)
    predicted_class = np.argmax(prediction)
    return class_names[predicted_class]

# Capture real-time video and predict using MediaPipe
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip the frame for natural gesture recognition and convert to RGB
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame using MediaPipe Hands
    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Draw hand landmarks
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)


            h,w, _ =frame.shape
            x_min=min([landmark.x for landmark in hand_landmarks.landmark])*w
            y_min=min([landmark.y for landmark in hand_landmarks.landmark])*h
            x_max=max([landmark.x for landmark in hand_landmarks.landmark])*w
            y_max=max([landmark.y for landmark in hand_landmarks.landmark])*h

            # Define region of interest (ROI) around the hand

            x_min, x_max = int(x_min), int(x_max)
            y_min, y_max = int(y_min), int(y_max)
            roi = frame[y_min:y_max, x_min:x_max]

            if roi.size > 0:
                # Predict the gesture from the ROI
                predicted_gesture = predict_sign(roi, model, class_names)

                # Draw the ROI rectangle and display prediction
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 255, 255), 2)
                cv2.putText(frame, f'Sign: {predicted_gesture}', (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Show the frame with landmarks and prediction
    cv2.imshow("Sign Language Recognition with MediaPipe", frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()